In [ ]:
import os
import re
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import matplotlib as mpl
from dataclasses import dataclass
from typing import Dict, Any, List, Optional, Tuple

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
VALID_GESTURES = {"OpenPalm", "TwoFingers"}
GESTURES_OF_INTEREST = {"OpenPalm", "TwoFingers"}

ROOT_DIR = 
OUT_DIR = 
os.makedirs(OUT_DIR, exist_ok=True)

COL_PETOUTPUTS_CANDIDATES = ["PETOutputs"]

GESTURE_ON = "OpenPalm"
GESTURE_OFF = {"TwoFingers", "Victory"}
DESIRED_OBF = {"OpenPalm": 1, "TwoFingers": 0}

TOGGLE_ON = "on"
TOGGLE_OFF = "off"

# Performance analysis

In [ ]:
def pipejson_to_dict(s: str) -> Optional[Dict[str, Any]]:
    if not isinstance(s, str) or not s.strip():
        return None

    parts, buf = [], []
    in_str = False
    esc = False
    depth = 0

    for ch in s:
        if esc:
            buf.append(ch); esc = False; continue
        if ch == "\\":
            buf.append(ch); esc = True; continue
        if ch == '"':
            in_str = not in_str
            buf.append(ch); continue

        if not in_str:
            if ch in "{[":
                depth += 1
            elif ch in "}]":
                depth -= 1

            # split only at top-level inside the root object
            if ch == "|" and depth == 1:
                parts.append("".join(buf))
                buf = []
                continue

        buf.append(ch)

    if buf:
        parts.append("".join(buf))

    js = ",".join(parts)
    try:
        return json.loads(js)
    except Exception:
        # last resort
        try:
            return json.loads(js.replace("|", ","))
        except Exception:
            return None

def _safe_float(x) -> Optional[float]:
    try:
        if x is None or (isinstance(x, float) and np.isnan(x)) or (isinstance(x, str) and not x.strip()):
            return None
        return float(x)
    except Exception:
        return None

In [ ]:
def _find_col(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    cols = {c.strip(): c for c in df.columns}
    for cand in candidates:
        if cand in cols:
            return cols[cand]
    # fallback: substring match
    for c in df.columns:
        if "pet" in c.lower() and "output" in c.lower():
            return c
    return None


In [ ]:
FOLDER_RE = re.compile(r"^(high|low)-vid-(\d+)$", re.IGNORECASE)

def discover_trials(root_dir: str) -> pd.DataFrame:
    rows = []
    for device_name in sorted(os.listdir(root_dir)):
        dev_path = os.path.join(root_dir, device_name)
        if not os.path.isdir(dev_path):
            continue

        for cond_folder in sorted(os.listdir(dev_path)):
            m = FOLDER_RE.match(cond_folder)
            if not m:
                continue
            stack = m.group(1).lower()      # high/low
            vid = f"vid{int(m.group(2))}"   # vid1, vid2, ...

            cond_path = os.path.join(dev_path, cond_folder)
            for fn in sorted(os.listdir(cond_path)):
                if not fn.lower().endswith(".csv"):
                    continue
                rows.append({
                    "device": device_name,
                    "stack": stack,
                    "video": vid,
                    "file": fn,
                    "path": os.path.join(cond_path, fn),
                })
    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError(f"No trials found. Check ROOT_DIR and folder naming. ROOT_DIR={root_dir}")
    return df

In [ ]:
trials_df = discover_trials(ROOT_DIR)
display(trials_df.head(24))

In [ ]:
def parse_trial_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    col_out = _find_col(df, COL_PETOUTPUTS_CANDIDATES)
    if col_out is None:
        raise ValueError(f"PETOutputs column not found in {path}. Columns={df.columns.tolist()}")

    col_fps = "FPS" if "FPS" in df.columns else None
    col_t = "ElapsedT(s)" if "ElapsedT(s)" in df.columns else None
    col_frame = "FrameNum" if "FrameNum" in df.columns else None

    frame_rows = []
    for i, row in df.iterrows():
        out = pipejson_to_dict(row[col_out])
        if out is None:
            continue

        frame_num = int(row[col_frame]) if col_frame else int(i)
        t_s = _safe_float(row[col_t]) if col_t else _safe_float(out.get("t_s"))
        fps = _safe_float(row[col_fps]) if col_fps else _safe_float(out.get("fps"))

        fr = {"frame": frame_num, "t_s": t_s, "fps": fps}

        # common timing fields (adjust if your keys differ)
        for k in ["ms_total", "ms_face", "ms_palm", "ms_pose", "ms_blur", "ms_misc"]:
            if k in out:
                fr[k] = _safe_float(out.get(k))

        frame_rows.append(fr)

    return pd.DataFrame(frame_rows).sort_values("frame").reset_index(drop=True)

In [ ]:
def trial_summary(frames: pd.DataFrame) -> Dict[str, Any]:
    out = {}
    for k in ["fps", "ms_total", "ms_face", "ms_palm", "ms_pose", "ms_blur"]:
        if k in frames.columns:
            s = frames[k].dropna().astype(float)
            if not s.empty:
                out[f"{k}_mean"] = float(s.mean())
                out[f"{k}_median"] = float(s.median())
    out["n_frames_parsed"] = int(len(frames))
    return out

In [ ]:
run_rows = []
for r in trials_df.itertuples(index=False):
    frames = parse_trial_csv(r.path)
    summ = trial_summary(frames)
    run_rows.append({
        "device": r.device,
        "stack": r.stack,
        "video": r.video,
        "file": r.file,
        "path": r.path,
        **summ
    })

In [ ]:
runs_df = pd.DataFrame(run_rows)
runs_df.to_csv(os.path.join(OUT_DIR, "runs_summary.csv"), index=False)
display(runs_df.head(24))

In [ ]:
METRICS = [c for c in runs_df.columns if c.endswith("_mean") and (c.startswith("fps") or c.startswith("ms_"))]

In [ ]:
agg = (runs_df
       .groupby(["device", "stack", "video"], as_index=False)
       .agg({m: ["mean", "std", "count"] for m in METRICS}))

In [ ]:
agg.columns = ["_".join([c for c in col if c]).rstrip("_") for col in agg.columns.values]
agg = agg.rename(columns={"device_": "device", "stack_": "stack", "video_": "video"})
agg.to_csv(os.path.join(OUT_DIR, "agg_device_stack_video.csv"), index=False)
display(agg)

In [ ]:
def savefig(fname: str):
    p = os.path.join(OUT_DIR, fname)
    plt.tight_layout()
    plt.savefig(p, dpi=200)
    print("[saved]", p)

In [ ]:
def plot_grouped_metric(
    agg_df,
    metric_base: str,
    title: str,
    ylabel: str,
    out_dir: str,
    fname: str,
    show: bool = True,

    title_fontsize: int = 14,
    xlabel: str = None,
    xlabel_fontsize: int = 12,
    ylabel_fontsize: int = 12,
    xtick_fontsize: int = 10,
    ytick_fontsize: int = 10,
    legend_fontsize: int = 10,
    figsize=(6.4, 4.8), 
    bar_width=0.3
):

    def normalize_device_name(dev: str) -> str:
        d = str(dev).lower()
        if "mq3" in d:
            return "MQ3"
        if "ml2" in d:
            return "ML2"
        if "hl2" in d:
            return "HL2"
        return str(dev)

    device_colors = {
        "HL2": "tab:orange",
        "MQ3": "green",
        "ML2": "#a6cee3",
    }

    video_label = {"vid1": "Video 1", "vid2": "Video 2"}
    stack_label = {"low": "Low", "high": "High"}

    y_col = f"{metric_base}_mean"
    e_col = f"{metric_base}_std"

    if y_col not in agg_df.columns:
        print(f"[skip] missing column: {y_col}")
        return None

    raw_devices = sorted(agg_df["device"].unique().tolist())

    desired_order = ["MQ3", "ML2"]
    order_idx = {name: i for i, name in enumerate(desired_order)}
    
    raw_devices = sorted(
        raw_devices,
        key=lambda d: order_idx.get(normalize_device_name(d), 999)
    )
    
    devices_pretty = [normalize_device_name(d) for d in raw_devices]

    stacks = ["high", "low"]
    videos = sorted(
        agg_df["video"].unique().tolist(),
        key=lambda x: int(str(x).replace("vid", "")) if str(x).startswith("vid") else str(x),
    )

    cats = [(v, s) for v in videos for s in stacks]
    x = np.arange(len(cats), dtype=float)

    width = bar_width
    offsets = (np.arange(len(raw_devices)) - (len(raw_devices) - 1) / 2.0) * width

    fig, ax = plt.subplots(figsize=figsize)

    for di, (raw_dev, pretty_dev) in enumerate(zip(raw_devices, devices_pretty)):
        ys, es = [], []
        for (v, s) in cats:
            sub = agg_df[
                (agg_df["device"] == raw_dev)
                & (agg_df["video"] == v)
                & (agg_df["stack"] == s)
            ]
            if sub.empty:
                ys.append(np.nan)
                es.append(0.0)
            else:
                ys.append(float(sub.iloc[0][y_col]))
                es.append(
                    float(sub.iloc[0][e_col])
                    if e_col in sub.columns and not np.isnan(sub.iloc[0][e_col])
                    else 0.0
                )

        ax.bar(
            x + offsets[di],
            ys,
            width=width,
            yerr=es,
            label=pretty_dev,
            color=device_colors.get(pretty_dev, None),
        )

    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"{video_label.get(v, v)}\n{stack_label.get(s, s)}" for (v, s) in cats],
        fontsize=xtick_fontsize,
    )

    if xlabel is not None:
        ax.set_xlabel(xlabel, fontsize=xlabel_fontsize, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=ylabel_fontsize, fontweight="bold")

    ax.tick_params(axis="y", labelsize=ytick_fontsize)

    ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.22),  
        ncol=len(devices_pretty),     
        fontsize=legend_fontsize,
        frameon=True,
    )
    fig.subplots_adjust(bottom=0.17) 

    fig.tight_layout()

    out_path = os.path.join(out_dir, fname)
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    print("[saved]", out_path)

    if show:
        plt.show()
    else:
        plt.close(fig)

    return fig

In [ ]:
metric_bases = [m.replace("_mean", "") for m in METRICS]

In [ ]:
metric_base_names = METRICS

In [ ]:
plot_grouped_metric(
    agg, metric_base="fps_mean",
    title="Performance (FPS) across visual stimuli and model stack configuration across headsets",
    ylabel="Average FPS",
    xlabel="Visual Stimulus / Model Stack",
    out_dir=OUT_DIR,
    fname="fps_by_stack_video_devices.pdf",
    title_fontsize=15,
    xlabel_fontsize=15,
    ylabel_fontsize=15,
    xtick_fontsize=15,
    ytick_fontsize=15,
    legend_fontsize=15,
    figsize=(6.4, 4.8),
    bar_width=0.3,
)

In [ ]:
plot_grouped_metric(
        agg, metric_base="ms_total_mean",
        title="End-to-end PET time per frame by Model Stack and Video (mean ± std across runs)",
        ylabel="ms/frame",
        out_dir=OUT_DIR,
        fname="ms_total_by_stack_video_devices.png",
    )

In [ ]:
def plot_grouped_stacked_stages_stagehatch_devicecolor(
    agg_df: pd.DataFrame,
    stage_bases: List[str],
    out_dir: str,
    fname: str,
    show: bool = True,

    xlabel: str = "Visual Stimulus / Model Stack",
    ylabel: str = "Time (milliseconds)",
    figsize=(6.4, 4.8),
    bar_width: float = 0.3,

    xlabel_fontsize: int = 15,
    ylabel_fontsize: int = 15,
    xtick_fontsize: int = 15,
    ytick_fontsize: int = 15,
    legend_fontsize: int = 15,
):

    def normalize_device_name(dev: str) -> str:
        d = str(dev).lower()
        if "mq3" in d:
            return "MQ3"
        if "ml2" in d:
            return "ML2"
        if "hl2" in d:
            return "HL2"
        return str(dev)

    device_colors = {
        "HL2": "tab:orange",
        "MQ3": "green",
        "ML2": "#a6cee3",
    }

    stage_to_hatch = {
        "ms_face": "////",   # diagonal
        "ms_palm": "....",   # dots
        "ms_pose": "xxxx",   # cross
        "ms_blur": "++++",   # plus grid
        "ms_misc": "OOOO",   # big circles (more distinct than 'oo')
    }

    def pretty_stage_name(st: str) -> str:
        name = st.replace("ms_", "")
        return name[:1].upper() + name[1:]  # Face, Palm, Pose, Blur, Misc

    video_label = {"vid1": "Video 1", "vid2": "Video 2"}
    stack_label = {"low": "Low", "high": "High"}

    stages_present = [st for st in stage_bases if f"{st}_mean_mean" in agg_df.columns]
    if not stages_present:
        print("[skip] None of the requested stages exist in agg_df.")
        return None

    raw_devices = agg_df["device"].unique().tolist()
    desired_order = ["MQ3", "ML2", "HL2"]
    order_idx = {name: i for i, name in enumerate(desired_order)}

    raw_devices = sorted(raw_devices, key=lambda d: order_idx.get(normalize_device_name(d), 999))
    devices_pretty = [normalize_device_name(d) for d in raw_devices]

    stacks = ["high", "low"]
    videos = sorted(
        agg_df["video"].unique().tolist(),
        key=lambda x: int(str(x).replace("vid", "")) if str(x).startswith("vid") else str(x),
    )

    cats = [(v, s) for v in videos for s in stacks]
    x = np.arange(len(cats), dtype=float)
    
    width = bar_width
    offsets = (np.arange(len(raw_devices)) - (len(raw_devices) - 1) / 2.0) * width

    fig, ax = plt.subplots(figsize=figsize)

    for di, (raw_dev, pretty_dev) in enumerate(zip(raw_devices, devices_pretty)):
        bottoms = np.zeros(len(cats), dtype=float)
        dev_color = device_colors.get(pretty_dev, None)

        for st in stages_present:
            mean_col = f"{st}_mean_mean"
            vals = []
            for (v, s) in cats:
                sub = agg_df[(agg_df["device"] == raw_dev) & (agg_df["video"] == v) & (agg_df["stack"] == s)]
                if sub.empty:
                    vals.append(0.0)
                else:
                    val = sub.iloc[0][mean_col]
                    vals.append(float(val) if pd.notna(val) else 0.0)

            vals = np.array(vals, dtype=float)

            ax.bar(
                x + offsets[di],
                vals,
                width=width,
                bottom=bottoms,
                color=dev_color,
                hatch=stage_to_hatch.get(st, ""),
                edgecolor="black",
                linewidth=0.8,
            )
            bottoms += vals

    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"{video_label.get(v, v)}\n{stack_label.get(s, s)}" for (v, s) in cats],
        fontsize=xtick_fontsize,
    )

    ax.set_xlabel(xlabel, fontsize=xlabel_fontsize, fontweight="bold", labelpad=6)
    ax.set_ylabel(ylabel, fontsize=ylabel_fontsize, fontweight="bold")
    ax.tick_params(axis="y", labelsize=ytick_fontsize)
    ymax = ax.get_ylim()[1]
    ax.set_yticks(np.arange(0, ymax + 1, 25))

    stage_handles = [
        Patch(
            facecolor="white",
            edgecolor="black",
            hatch=stage_to_hatch.get(st, ""),
            label=pretty_stage_name(st),
            linewidth=1.0,
        )
        for st in stages_present
    ]
    
    device_handles = [
        Patch(facecolor=device_colors.get(pretty, "white"), edgecolor="black", label=pretty)
        for pretty in devices_pretty
    ]
    
    fig.tight_layout(rect=[0.02, 0.14, 0.98, 0.88])
    pos = ax.get_position()
    
    leg_stage = fig.legend(
        handles=stage_handles,
        loc="lower left",
        bbox_to_anchor=(pos.x0 - 0.017, pos.y1 + 0.015),  # right + up
        bbox_transform=fig.transFigure,
        ncol=len(stage_handles),
        fontsize=legend_fontsize,
        frameon=True,
        columnspacing=1.2,
        handlelength=2.0,
    )
    leg_stage.set_title("Module", prop={"size": legend_fontsize})
    
    fig.legend(
        handles=device_handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.070),  # was 0.055
        ncol=len(device_handles),
        fontsize=legend_fontsize,
        frameon=True,
    )

    out_path = os.path.join(out_dir, fname)
    fig.savefig(out_path, dpi=200, bbox_inches="tight", pad_inches=0.06)
    print("[saved]", out_path)

    if show:
        plt.show()
    else:
        plt.close(fig)

    return fig

In [ ]:
stages = ["ms_face", "ms_palm", "ms_pose", "ms_blur", "ms_misc"]

mpl.rcParams["hatch.linewidth"] = 1.4   # try 1.2–2.0

plot_grouped_stacked_stages_stagehatch_devicecolor(
    agg_df=agg,
    stage_bases=stages,
    out_dir=OUT_DIR,
    fname="ms_stacked_modules_devicecolor_stagehatch.pdf",
    show=True,
    figsize=(6.4, 4.8),
    bar_width=0.4,
    xlabel_fontsize=15,
    ylabel_fontsize=15,
    xtick_fontsize=15,
    ytick_fontsize=15,
    legend_fontsize=15,
)

# Correctness

In [ ]:
def pipejson_to_dict(s: str):
    if not isinstance(s, str) or not s.strip():
        return None
    parts, buf = [], []
    in_str = False
    esc = False
    for ch in s:
        if esc:
            buf.append(ch)
            esc = False
            continue
        if ch == "\\":
            buf.append(ch)
            esc = True
            continue
        if ch == '"':
            in_str = not in_str
            buf.append(ch)
            continue
        if ch == "|" and not in_str:
            parts.append("".join(buf))
            buf = []
        else:
            buf.append(ch)
    parts.append("".join(buf))
    try:
        return json.loads(",".join(parts))
    except:
        return None


def infer_from_path(csv_path, root_dir):
    path = Path(csv_path).resolve()
    root = Path(root_dir).resolve()

    rel = path.relative_to(root)
    parts = rel.parts

    device = parts[0]

    cond_vid_folder = parts[1].lower()

    if cond_vid_folder.startswith("high"):
        condition = "high"
    elif cond_vid_folder.startswith("low"):
        condition = "low"
    else:
        condition = "unknown"

    if "vid-1" in cond_vid_folder:
        video = "vid1"
    elif "vid-2" in cond_vid_folder:
        video = "vid2"
    else:
        video = "unknown"

    return device, condition, video

def load_faces(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]

    if "PETOutputs" not in df.columns:
        raise ValueError("PETOutputs column missing")

    parsed = df["PETOutputs"].astype(str).apply(pipejson_to_dict)

    rows = []
    for i, d in enumerate(parsed):
        if not d:
            continue
        faces = d.get("faces", [])
        if not isinstance(faces, list):
            continue
        for f in faces:
            rows.append({
                "frame": i,
                "face_id": int(float(f.get("id", -1))),
                "obf": int(float(f.get("obf", 0))),
            })

    return pd.DataFrame(rows)


def compute_toggle_stats(face_df):
    if len(face_df) == 0:
        return None, None, 0

    g = face_df.sort_values("frame")
    obf = g["obf"].to_numpy()

    initial = int(obf[0])
    final = int(obf[-1])

    toggles = np.sum(obf[1:] != obf[:-1])

    return initial, final, int(toggles)


def evaluate_trial(csv_path):
    device, condition, video = infer_from_path(csv_path, ROOT_DIR)
    faces_df = load_faces(csv_path)

    if len(faces_df) == 0:
        return {
            "csv_path": csv_path,
            "device": device,
            "condition": condition,
            "video": video,
            "trial_pass": False,
            "reason": "no_faces_detected",
            "reason_detail": "No faces were detected in this trial."
        }

    face_ids = faces_df["face_id"].unique().tolist()

    if video == "vid1":
        fid = face_ids[0]
        fdf = faces_df[faces_df["face_id"] == fid]

        init, final, toggles = compute_toggle_stats(fdf)

        if init != 0:
            return {
                "device": device, "csv_path": csv_path,
                "condition": condition, "video": video,
                "trial_pass": False,
                "reason": "wrong_initial",
                "reason_detail": f"Initial obf state was {init}, expected 0 (OFF)."
            }

        if final != 0:
            return {
                "device": device, "csv_path": csv_path,
                "condition": condition, "video": video,
                "trial_pass": False,
                "reason": "wrong_final",
                "reason_detail": f"Final obf state was {final}, expected 0 (OFF)."
            }

        if toggles != 2:
            return {
                "device": device, "csv_path": csv_path,
                "condition": condition, "video": video,
                "trial_pass": False,
                "reason": "wrong_toggle_count",
                "reason_detail": f"Observed {toggles} obf toggles, expected exactly 2 (OFF→ON→OFF)."
            }

        return {
            "device": device, "csv_path": csv_path,
            "condition": condition, "video": video,
            "trial_pass": True,
            "reason": "pass",
            "reason_detail": "Initial and final obf states correct; observed expected OFF→ON→OFF toggle sequence."
        }

    else:
        toggle_counts = {}
        for fid in face_ids:
            fdf = faces_df[faces_df["face_id"] == fid]
            _, _, t = compute_toggle_stats(fdf)
            toggle_counts[fid] = t

        actor_id = max(toggle_counts, key=toggle_counts.get)
        non_actor_ids = [f for f in face_ids if f != actor_id]

        actor_df = faces_df[faces_df["face_id"] == actor_id]
        init_a, final_a, toggles_a = compute_toggle_stats(actor_df)

        # Actor checks
        if init_a != 0:
            return {
                "device": device, "csv_path": csv_path,
                "condition": condition, "video": video,
                "trial_pass": False,
                "reason": "actor_wrong_initial",
                "reason_detail": f"Actor initial obf was {init_a}, expected 0 (OFF)."
            }

        if final_a != 0:
            return {
                "device": device, "csv_path": csv_path,
                "condition": condition, "video": video,
                "trial_pass": False,
                "reason": "actor_wrong_final",
                "reason_detail": f"Actor final obf was {final_a}, expected 0 (OFF)."
            }

        if toggles_a != 2:
            return {
                "device": device, "csv_path": csv_path,
                "condition": condition, "video": video,
                "trial_pass": False,
                "reason": "actor_wrong_toggle_count",
                "reason_detail": f"Actor had {togles_a} toggles, expected exactly 2 (OFF→ON→OFF)."
            }

        # Non-actor must never toggle
        for fid in non_actor_ids:
            fdf = faces_df[faces_df["face_id"] == fid]
            _, _, t = compute_toggle_stats(fdf)
            if t != 0:
                return {
                    "device": device, "csv_path": csv_path,
                    "condition": condition, "video": video,
                    "trial_pass": False,
                    "reason": "non_actor_toggled",
                    "reason_detail": f"Non-actor face_id {fid} toggled {t} times but should remain OFF throughout."
                }

        return {
            "device": device, "csv_path": csv_path,
            "condition": condition, "video": video,
            "trial_pass": True,
            "reason": "pass",
            "reason_detail": "Actor followed OFF→ON→OFF pattern; non-actor remained OFF."
        }

def get_all_csvs(root_dir):
    return [str(p) for p in Path(root_dir).rglob("*DataReplay*.csv")]


results = []
for csv_path in get_all_csvs(ROOT_DIR):
    try:
        results.append(evaluate_trial(csv_path))
    except Exception as e:
        device, condition, video = infer_from_path(csv_path, ROOT_DIR)
        results.append({
            "csv_path": csv_path,
            "device": device,
            "condition": condition,
            "video": video,
            "trial_pass": False,
            "reason": f"error:{type(e).__name__}",
            "reason_detail": f"Exception occurred during evaluation: {str(e)}"
        })

summary_df = pd.DataFrame(results)
summary_df.to_csv(Path(OUT_DIR) / "trial_toggle_summary.csv", index=False)

print("Saved:", Path(OUT_DIR) / "trial_toggle_summary.csv")
print("\nPass rate:")
print(summary_df.groupby(["device","condition","video"])["trial_pass"].mean())

In [ ]:
summary_df["pass_numeric"] = summary_df["trial_pass"].astype(int)

# Compute mean and std per device × condition × video
stats = (
    summary_df.groupby(["device","condition","video"])["pass_numeric"]
              .agg(["mean","count"])
              .reset_index()
)

stats["mean_pct"] = stats["mean"] * 100

# Proper binomial standard error
stats["std_pct"] = (
    np.sqrt(stats["mean"] * (1 - stats["mean"]) / stats["count"])
) * 100



def normalize_device_name(dev: str) -> str:
    d = str(dev).lower()
    if "mq3" in d:
        return "MQ3"
    if "ml2" in d:
        return "ML2"
    if "hl2" in d:
        return "HL2"
    return str(dev)

FONT = 15
figsize = (6.4, 4.8)
bar_width = 0.3

device_colors = {
    "HL2": "tab:orange",
    "MQ3": "green",
    "ML2": "#a6cee3",
}

video_label = {"vid1": "Video 1", "vid2": "Video 2"}
cond_label  = {"high": "High", "low": "Low"}

raw_devices = stats["device"].unique().tolist()
desired_order = ["MQ3", "ML2", "HL2"]
order_idx = {name: i for i, name in enumerate(desired_order)}
raw_devices = sorted(raw_devices, key=lambda d: order_idx.get(normalize_device_name(d), 999))
devices_pretty = [normalize_device_name(d) for d in raw_devices]

videos = ["vid1", "vid2"]
conditions = ["high", "low"]
cats = [(v, c) for v in videos for c in conditions]

labels = [f"{video_label.get(v, v)}\n{cond_label.get(c, c)}" for (v, c) in cats]
x = np.arange(len(labels), dtype=float)

means = {d: [] for d in raw_devices}
errs  = {d: [] for d in raw_devices}

for (vid, cond) in cats:
    for d in raw_devices:
        row = stats[
            (stats["device"] == d) &
            (stats["video"] == vid) &
            (stats["condition"] == cond)
        ]
        if len(row):
            means[d].append(float(row["mean_pct"].iloc[0]))
            errs[d].append(float(row["std_pct"].iloc[0]) if not np.isnan(row["std_pct"].iloc[0]) else 0.0)
        else:
            means[d].append(0.0)
            errs[d].append(0.0)

fig, ax = plt.subplots(figsize=figsize)

width = bar_width
offsets = (np.arange(len(raw_devices)) - (len(raw_devices) - 1) / 2.0) * width

for di, (raw_dev, pretty_dev) in enumerate(zip(raw_devices, devices_pretty)):
    ax.bar(
        x + offsets[di],
        means[raw_dev],
        width=width,
        yerr=errs[raw_dev],
        capsize=4,
        label=pretty_dev,
        color=device_colors.get(pretty_dev, None),
    )

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=FONT)

ax.set_xlabel("Visual Stimulus / Model Stack", fontsize=FONT, fontweight="bold", labelpad=10)
ax.set_ylabel("Pass Rate (%)", fontsize=FONT, fontweight="bold")
ax.tick_params(axis="y", labelsize=FONT)

ax.set_ylim(0, 110)

ax.set_yticks(np.arange(0, 111, 25))

ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.26),
    ncol=len(devices_pretty),
    fontsize=FONT,
    frameon=True,
)

fig.subplots_adjust(bottom=0.26)
fig.savefig(Path(OUT_DIR) / "toggle_passrate_by_stack_video_devices.pdf", dpi=200, bbox_inches="tight")
plt.show()

# Responsiveness

In [ ]:
def normalize_device_name(dev: str) -> str:
    d = str(dev).lower()
    if "mq3" in d:
        return "MQ3"
    if "ml2" in d:
        return "ML2"
    if "hl2" in d:
        return "HL2"
    return str(dev)

def plot_device_grouped_bars_from_stats(
    plot_stats,                
    title: str,
    ylabel: str,
    out_path: str,
    xlabel: str = "Visual Stimuli / Model Stack",
    figsize=(6.4, 4.8),
    bar_width=0.3,
    fontsize=15,
    ytick_step=25,             
    show=True,
):

    device_colors = {
        "HL2": "tab:orange",
        "MQ3": "green",
        "ML2": "#a6cee3",
    }

    video_label = {"vid1": "Video 1", "vid2": "Video 2"}
    cond_label  = {"high": "High", "low": "Low"}

    raw_devices = plot_stats["device"].unique().tolist()
    desired_order = ["MQ3", "ML2", "HL2"]
    order_idx = {name: i for i, name in enumerate(desired_order)}
    raw_devices = sorted(raw_devices, key=lambda d: order_idx.get(normalize_device_name(d), 999))
    devices_pretty = [normalize_device_name(d) for d in raw_devices]

    videos = ["vid1", "vid2"]
    conditions = ["high", "low"]
    cats = [(v, c) for v in videos for c in conditions]
    x = np.arange(len(cats), dtype=float)

    means = {d: [] for d in raw_devices}
    stds  = {d: [] for d in raw_devices}

    for (vid, cond) in cats:
        for d in raw_devices:
            row = plot_stats[
                (plot_stats["device"] == d) &
                (plot_stats["video"] == vid) &
                (plot_stats["condition"] == cond)
            ]
            if len(row):
                m = float(row["mean"].iloc[0])
                s = row["std"].iloc[0]
                s = float(s) if (s is not None and not np.isnan(s)) else 0.0
            else:
                m, s = 0.0, 0.0
            means[d].append(m)
            stds[d].append(s)

    fig, ax = plt.subplots(figsize=figsize)

    width = bar_width
    offsets = (np.arange(len(raw_devices)) - (len(raw_devices) - 1) / 2.0) * width

    for di, (raw_dev, pretty_dev) in enumerate(zip(raw_devices, devices_pretty)):
        ax.bar(
            x + offsets[di],
            means[raw_dev],
            width=width,
            yerr=stds[raw_dev],
            capsize=4,
            label=pretty_dev,
            color=device_colors.get(pretty_dev, None),
        )

    xticklabels = [f"{video_label.get(v, v)}\n{cond_label.get(c, c)}" for (v, c) in cats]
    ax.set_xticks(x)
    ax.set_xticklabels(xticklabels, fontsize=fontsize)

    ax.set_xlabel(xlabel, fontsize=fontsize, fontweight="bold", labelpad=10)
    ax.set_ylabel(ylabel, fontsize=fontsize, fontweight="bold")
    ax.tick_params(axis="y", labelsize=fontsize)

    ymax = ax.get_ylim()[1]
    ax.set_yticks(np.arange(0, ymax + 1, ytick_step))

    ax.legend(
        loc="upper center",
        bbox_to_anchor=(0.5, -0.26),
        ncol=len(devices_pretty),
        fontsize=fontsize,
        frameon=True,
    )

    fig.subplots_adjust(bottom=0.26)
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    print("[saved]", out_path)

    if show:
        plt.show()
    else:
        plt.close(fig)

    return fig


def pipejson_to_dict(s: str):
    if not isinstance(s, str) or not s.strip():
        return None
    parts, buf = [], []
    in_str = False
    esc = False
    for ch in s:
        if esc:
            buf.append(ch)
            esc = False
            continue
        if ch == "\\":
            buf.append(ch)
            esc = True
            continue
        if ch == '"':
            in_str = not in_str
            buf.append(ch)
            continue
        if ch == "|" and not in_str:
            parts.append("".join(buf))
            buf = []
        else:
            buf.append(ch)
    parts.append("".join(buf))
    try:
        return json.loads(",".join(parts))
    except:
        return None

def infer_from_path(csv_path: str, root_dir: str):
    path = Path(csv_path).resolve()
    root = Path(root_dir).resolve()
    rel = path.relative_to(root)
    parts = rel.parts

    device = parts[0] if len(parts) >= 1 else "UnknownDevice"
    cond_vid = parts[1].lower() if len(parts) >= 2 else "unknown"

    if cond_vid.startswith("high"):
        condition = "high"
    elif cond_vid.startswith("low"):
        condition = "low"
    else:
        condition = "unknown"

    if "vid-1" in cond_vid:
        video = "vid1"
    elif "vid-2" in cond_vid:
        video = "vid2"
    else:
        video = "unknown"

    return device, condition, video


def find_trial_csvs(root_dir: str):
    return [str(p) for p in Path(root_dir).rglob("*DataReplay*.csv")]


def _safe_float(x, default=np.nan):
    try:
        if x is None:
            return default
        s = str(x).strip()
        if s == "":
            return default
        return float(s)
    except:
        return default


def _safe_int(x, default=-1):
    try:
        if x is None:
            return default
        s = str(x).strip()
        if s == "":
            return default
        return int(float(s))
    except:
        return default


def load_trial_frames_and_faces(csv_path: str):
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]

    rename = {}
    for c in df.columns:
        cs = c.strip()
        if cs in ["ElapsedTime", "ElapsedT(s)", "ElapsedT (s)", "ElapsedTime(ms)", "ElapsedTime (ms)", "ElapsedTimeMS", "ElapsedTimeMs", "ElapsedTimeMillis", "ElapsedTimeMillisec", "ElapsedTimeMilliseconds", "ElapsedTimeMilliseconds(ms)", "ElapsedTimeMilliseconds (ms)"]:
            rename[c] = "ElapsedTime"
        if cs in ["ElapsedTime", "ElapsedTime(ms)", "ElapsedTime (ms)"]:
            rename[c] = "ElapsedTime"
        if cs == "ElapsedTime" or cs == "ElapsedTime(ms)" or cs == "ElapsedTime (ms)":
            rename[c] = "ElapsedTime"
        if cs == "frame #" or cs == "frame" or cs == "FrameNum":
            rename[c] = "frame"
        if cs == "FPS":
            rename[c] = "FPS"
        if cs == "PETOutputs":
            rename[c] = "PETOutputs"
    df = df.rename(columns=rename)

    if "PETOutputs" not in df.columns:
        for c in df.columns:
            if c.replace(" ", "") == "PETOutputs":
                df = df.rename(columns={c: "PETOutputs"})
                break

    if "PETOutputs" not in df.columns:
        raise ValueError(f"PETOutputs column missing in {csv_path}")

    if "frame" not in df.columns:
        for c in df.columns:
            if c.replace(" ", "") in ("frame#", "frame", "FrameNum"):
                df = df.rename(columns={c: "frame"})
                break
    if "frame" not in df.columns:
        df["frame"] = np.arange(len(df), dtype=int)
    df["frame"] = pd.to_numeric(df["frame"], errors="coerce")
    df["frame"] = df["frame"].fillna(pd.Series(np.arange(len(df)), index=df.index)).astype(int)

    # elapsed time (ms)
    if "ElapsedTime" not in df.columns:
        for c in df.columns:
            if c.replace(" ", "") == "ElapsedTime":
                df = df.rename(columns={c: "ElapsedTime"})
                break
    if "ElapsedTime" not in df.columns:
        df["ElapsedTime"] = 0.0
    df["ElapsedTime"] = pd.to_numeric(df["ElapsedTime"], errors="coerce").fillna(0.0).astype(float)

    # fps
    if "FPS" not in df.columns:
        for c in df.columns:
            if c.replace(" ", "") == "FPS":
                df = df.rename(columns={c: "FPS"})
                break
    if "FPS" not in df.columns:
        df["FPS"] = 0.0
    df["FPS"] = pd.to_numeric(df["FPS"], errors="coerce").fillna(0.0).astype(float)

    if df["ElapsedTime"].max() <= 0 or df["ElapsedTime"].nunique() <= 1:
        fps = np.clip(df["FPS"].to_numpy(dtype=float), 1.0, 120.0)
        dt_ms = 1000.0 / fps
        df["ElapsedTime"] = np.cumsum(dt_ms) - dt_ms[0]

    parsed = df["PETOutputs"].astype(str).apply(pipejson_to_dict)

    frame_rows = []
    face_rows = []

    for i, d in enumerate(parsed):
        if not d:
            continue

        t_ms = float(df.loc[i, "ElapsedTime"])
        frame_idx = int(df.loc[i, "frame"])

        ms_face = _safe_float(d.get("ms_face", np.nan))
        ms_palm = _safe_float(d.get("ms_palm", np.nan))
        ms_pose = _safe_float(d.get("ms_pose", np.nan))
        ms_blur = _safe_float(d.get("ms_blur", np.nan))
        ms_total = _safe_float(d.get("ms_total", np.nan))

        frame_rows.append({
            "csv_path": csv_path,
            "frame": frame_idx,
            "t_ms": t_ms,
            "ms_face": ms_face,
            "ms_palm": ms_palm,
            "ms_pose": ms_pose,
            "ms_blur": ms_blur,
            "ms_total": ms_total,
        })

        faces = d.get("faces", [])
        if isinstance(faces, list):
            for f in faces:
                face_rows.append({
                    "csv_path": csv_path,
                    "frame": frame_idx,
                    "t_ms": t_ms,
                    "face_id": _safe_int(f.get("id", -1)),
                    "palm": _safe_int(f.get("palm", -1)),
                    "gesture": str(f.get("g", "Unknown")),
                    "obf": _safe_int(f.get("obf", 0)),
                })

    frames_df = pd.DataFrame(frame_rows)
    faces_df = pd.DataFrame(face_rows)

    if len(frames_df) == 0:
        frames_df = pd.DataFrame(columns=["csv_path","frame","t_ms","ms_face","ms_palm","ms_pose","ms_blur","ms_total"])
    if len(faces_df) == 0:
        faces_df = pd.DataFrame(columns=["csv_path","frame","t_ms","face_id","palm","gesture","obf"])

    return frames_df, faces_df


def pick_actor_face_id(faces_df: pd.DataFrame):
    if len(faces_df) == 0:
        return -1
    cmd = faces_df[(faces_df["gesture"].isin(["OpenPalm","TwoFingers"])) & (faces_df["palm"] != -1)]
    if len(cmd) > 0:
        return int(cmd["face_id"].value_counts().index[0])
    return int(faces_df["face_id"].value_counts().index[0])


def analysis3_module_latency(root_dir: str, out_dir: str, max_wait_ms: float = 2000.0):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    csvs = find_trial_csvs(root_dir)
    print(f"Found {len(csvs)} trial CSVs")

    event_rows = []
    trial_rows = []

    for csv_path in csvs:
        device, condition, video = infer_from_path(csv_path, root_dir)

        try:
            frames_df, faces_df = load_trial_frames_and_faces(csv_path)

            if len(frames_df) == 0 or len(faces_df) == 0:
                trial_rows.append({
                    "device": device, "condition": condition, "video": video, "csv_path": csv_path,
                    "actor_face_id": -1,
                    "n_events": 0,
                    "lat_sum_mean": np.nan,
                    "lat_sum_median": np.nan,
                    "lat_sum_p90": np.nan,
                    "ms_total_mean": np.nan,
                    "ms_total_median": np.nan,
                    "ms_total_p90": np.nan,
                    "error": "no_frames_or_faces"
                })
                continue

            actor_id = pick_actor_face_id(faces_df)
            f = faces_df[faces_df["face_id"] == actor_id].sort_values(["t_ms","frame"]).copy()

            f["valid_cmd"] = f["gesture"].isin(["OpenPalm","TwoFingers"]) & (f["palm"] != -1)
            f["prev_valid"] = f["valid_cmd"].shift(1).fillna(False).astype(bool)
            f["onset"] = f["valid_cmd"] & (~f["prev_valid"])

            onsets = f[f["onset"]][["frame","t_ms","gesture","obf","palm"]].copy()

            merged = onsets.merge(
                frames_df[["frame","ms_face","ms_palm","ms_pose","ms_blur","ms_total"]],
                on="frame",
                how="left"
            )

            merged["ms_sum_modules"] = (
                merged["ms_face"].fillna(0) +
                merged["ms_palm"].fillna(0) +
                merged["ms_pose"].fillna(0) +
                merged["ms_blur"].fillna(0)
            )

            for _, r in merged.iterrows():
                event_rows.append({
                    "device": device,
                    "condition": condition,
                    "video": video,
                    "csv_path": csv_path,
                    "actor_face_id": actor_id,
                    "frame_event": int(r["frame"]),
                    "t_ms_event": float(r["t_ms"]),
                    "gesture": str(r["gesture"]),
                    "palm": int(r["palm"]),
                    "obf_at_event": int(r["obf"]),
                    "ms_face": float(r["ms_face"]) if pd.notna(r["ms_face"]) else np.nan,
                    "ms_palm": float(r["ms_palm"]) if pd.notna(r["ms_palm"]) else np.nan,
                    "ms_pose": float(r["ms_pose"]) if pd.notna(r["ms_pose"]) else np.nan,
                    "ms_blur": float(r["ms_blur"]) if pd.notna(r["ms_blur"]) else np.nan,
                    "ms_sum_modules": float(r["ms_sum_modules"]),
                    "ms_total": float(r["ms_total"]) if pd.notna(r["ms_total"]) else np.nan,
                })

            lat = merged["ms_sum_modules"].to_numpy(dtype=float) if len(merged) else np.array([])
            tot = merged["ms_total"].dropna().to_numpy(dtype=float) if len(merged) else np.array([])

            trial_rows.append({
                "device": device, "condition": condition, "video": video, "csv_path": csv_path,
                "actor_face_id": actor_id,
                "n_events": int(len(merged)),
                "lat_sum_mean": float(np.mean(lat)) if len(lat) else np.nan,
                "lat_sum_median": float(np.median(lat)) if len(lat) else np.nan,
                "lat_sum_p90": float(np.percentile(lat, 90)) if len(lat) else np.nan,
                "ms_total_mean": float(np.mean(tot)) if len(tot) else np.nan,
                "ms_total_median": float(np.median(tot)) if len(tot) else np.nan,
                "ms_total_p90": float(np.percentile(tot, 90)) if len(tot) else np.nan,
                "error": ""
            })

        except Exception as e:
            trial_rows.append({
                "device": device, "condition": condition, "video": video, "csv_path": csv_path,
                "actor_face_id": -1,
                "n_events": 0,
                "lat_sum_mean": np.nan,
                "lat_sum_median": np.nan,
                "lat_sum_p90": np.nan,
                "ms_total_mean": np.nan,
                "ms_total_median": np.nan,
                "ms_total_p90": np.nan,
                "error": f"{type(e).__name__}: {e}"
            })

    events_df = pd.DataFrame(event_rows)
    summary_df = pd.DataFrame(trial_rows)

    events_path = out_dir / "analysis3_module_latency_events.csv"
    summary_path = out_dir / "analysis3_module_latency_summary.csv"
    events_df.to_csv(events_path, index=False)
    summary_df.to_csv(summary_path, index=False)

    print("Wrote:")
    print(" -", events_path)
    print(" -", summary_path)

    if len(events_df) == 0:
        print("No events found; skipping plot.")
        return events_df, summary_df

    trial_metric = (
        events_df.groupby(["device","condition","video","csv_path"])["ms_sum_modules"]
                 .median()
                 .reset_index(name="trial_median_ms_sum_modules")
    )

    plot_stats = (
        trial_metric.groupby(["device","condition","video"])["trial_median_ms_sum_modules"]
                    .agg(["mean","std","count"])
                    .reset_index()
    )

    devices = sorted(plot_stats["device"].unique())
    videos = ["vid1","vid2"]
    conditions = ["high","low"]

    labels = []
    means = {d: [] for d in devices}
    stds = {d: [] for d in devices}

    for vid in videos:
        for cond in conditions:
            labels.append(f"{vid}\n{cond}")
            for d in devices:
                row = plot_stats[
                    (plot_stats["device"] == d) &
                    (plot_stats["video"] == vid) &
                    (plot_stats["condition"] == cond)
                ]
                if len(row):
                    means[d].append(float(row["mean"].iloc[0]))
                    s = row["std"].iloc[0]
                    stds[d].append(float(s) if not np.isnan(s) else 0.0)
                else:
                    means[d].append(0.0)
                    stds[d].append(0.0)

    x = np.arange(len(labels))
    width = 0.35
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

    plot_path = out_dir / "analysis3_module_latency_bar.pdf"

    plot_device_grouped_bars_from_stats(
        plot_stats=plot_stats,
        title="Analysis 3: Pipeline Processing Cost at First Valid Gesture Onsets",
        ylabel="Time (milliseconds)",
        xlabel="Visual Stimulus / Model Stack",
        out_path=str(plot_path),
        figsize=(6.4, 4.8),
        bar_width=0.3,
        fontsize=15,
        ytick_step=50,
        show=True,
    )

    return events_df, summary_df

In [ ]:
def analysis_3():
    csvs = find_trial_csvs(ROOT_DIR)
    print(f"Found {len(csvs)} trial CSVs")

    all_events = []
    all_summaries = []

    for p in csvs:
        try:
            ev, summ = compute_latency_events_for_trial(p)
            all_events.append(ev)
            all_summaries.append(summ)
        except Exception as e:
            device, condition, video = infer_from_path(p, ROOT_DIR)
            all_summaries.append({
                "device": device,
                "condition": condition,
                "video": video,
                "csv_path": p,
                "actor_face_id": -1,
                "n_events": 0,
                "n_success": 0,
                "n_missed": 0,
                "lat_ms_mean": np.nan,
                "lat_ms_median": np.nan,
                "lat_ms_p90": np.nan,
                "error": f"{type(e).__name__}: {e}",
            })

    events_df = pd.concat(all_events, ignore_index=True) if len(all_events) else pd.DataFrame()
    summary_df = pd.DataFrame(all_summaries)

    events_path = Path(OUT_DIR) / "analysis3_latency_events.csv"
    summary_path = Path(OUT_DIR) / "analysis3_latency_summary.csv"

    events_df.to_csv(events_path, index=False)
    summary_df.to_csv(summary_path, index=False)

    print("Wrote:")
    print(" -", events_path)
    print(" -", summary_path)

    if len(events_df) == 0:
        return
    
    if "missed" not in events_df.columns:
        if "latency_ms" in events_df.columns:
            events_df["missed"] = events_df["latency_ms"].isna().astype(int)
        elif "t_reached_ms" in events_df.columns:
            events_df["missed"] = events_df["t_reached_ms"].isna().astype(int)
        else:
            return
            
    per_trial_lat = events_df[events_df["missed"] == 0].copy()

    if len(per_trial_lat) == 0:
        print("No successful events found; skipping plot.")
        return

    trial_med = (
        per_trial_lat.groupby(["device","condition","video","csv_path"])["latency_ms"]
                    .median()
                    .reset_index(name="trial_median_latency_ms")
    )

    plot_stats = (
        trial_med.groupby(["device","condition","video"])["trial_median_latency_ms"]
                 .agg(["mean","std","count"])
                 .reset_index()
    )

    devices = sorted(plot_stats["device"].unique())
    videos = ["vid1", "vid2"]
    conditions = ["high", "low"]

    labels = []
    means = {d: [] for d in devices}
    stds = {d: [] for d in devices}

    for vid in videos:
        for cond in conditions:
            labels.append(f"{vid}\n{cond}")
            for d in devices:
                row = plot_stats[
                    (plot_stats["device"] == d) &
                    (plot_stats["video"] == vid) &
                    (plot_stats["condition"] == cond)
                ]
                if len(row):
                    means[d].append(float(row["mean"].iloc[0]))
                    stds[d].append(float(row["std"].iloc[0]) if not np.isnan(row["std"].iloc[0]) else 0.0)
                else:
                    means[d].append(0.0)
                    stds[d].append(0.0)

    x = np.arange(len(labels))
    width = 0.35
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

    plot_path = Path(OUT_DIR) / "analysis3_latency_bar.png"

    plot_device_grouped_bars_from_stats(
        plot_stats=plot_stats,
        title="Analysis 3: Responsiveness by Model Stack and Video",
        ylabel="Gesture→Obfuscation Latency (ms)",
        xlabel="Visual Stimuli / Model Stack",
        out_path=str(plot_path),
        figsize=(6.4, 4.8),
        bar_width=0.3,
        fontsize=15,
        ytick_step=50,
        show=True,
    )

In [ ]:
events3_df, summary3_df = analysis3_module_latency(ROOT_DIR, OUT_DIR, max_wait_ms=2000.0)